## Cosmic Web - Intro

### Table of Contents

1. [Motivation](#1-motivation)
2. [Density field](#2-density-field)
3. [Why this problem](#3-why-this-problem)
4. [Algorithm](#4-algorithm)
5. [Reference](#5-reference)
6. [Validation](#6-validation)
7. [Sizing](#7-sizing)
8. [Baseline](#8-baseline)
9. [Ladder](#9-ladder)
10. [Conventions](#10-conventions)

---

### 1. Motivation

We have looked at array-oriented computing with NumPy and CuPy. Other libraries provide different programming models, and some of them suit a given scientific problem better. In this block we take one computation from [CuPy](https://docs.cupy.dev/en/stable/)'s array model through three more programming models: [JAX](https://docs.jax.dev/en/latest/)'s traced and fused graph, [cuda.compute](https://nvidia.github.io/cccl/python/)'s algorithms over iterators, and bindings to existing C++ kernels/libraries with [CppJIT](https://github.com/compiler-research/cppjit).

The computation comes from cosmology and is small enough to understand completely in this notebook.

### 2. Density field

A box holds one point at every corner of a grid, each a parcel of matter, with faint ripples in the mass. Gravity pulls every point towards the nearby excess. The Zel'dovich approximation ([Zel'dovich 1970](https://ui.adsabs.harvard.edu/abs/1970A%26A.....5...84Z)) is the simplest model of what follows:

- Each point keeps the direction of its first pull and moves along it in a straight line.
- The distance grows with time: $D$ times the first displacement, where the growth factor $D$ stands in for time.
- By $D = 1$ the first points have crossed each other, so the model stops there.

The animation follows the particles that started on one plane of the grid, seen from above, as $D$ runs from 0 to 1.


In [ ]:
import numpy as np
import cupy as cp

import cosmic_core as cw

cw.machine_report()
frames = cw.zeldovich_particles(n=128, start=0.0, end=1.0, interval=0.1)
cw.animate_particles(frames, n=128)

- At $D = 0$ the points sit on their grid. As $D$ grows they drift towards the regions that started out denser and pile up along lines.
- In 3D these lines are **sheets**, seen edge-on.
- Sheets meet along **filaments**, and filaments meet at **knots**, seen here as bright spots.
- The empty regions between them are **voids**.

This network of sheets, filaments and knots around voids is the **cosmic web** ([Bond et al. 1996](https://doi.org/10.1038/380603a0)).

A grid over the box turns the cloud into a *density field* in four steps, shown on one plane of the box at $D = 1$:

1. **Deposit.** Each grid cell stores the mass of the particles that landed in it.
2. **Contrast.** Divide by the mean and subtract one. The result $\delta$: positive where a cell holds more than the average and negative where it holds less. An empty cell has $\delta = -1$.
3. **Blur.** Smooth $\delta$ over a few cells. This removes the graininess of individual points and retains the structure.
4. **Standardise.** Divide the blurred $\delta$ by its scatter $\sigma$. The value $\nu = \delta/\sigma$ is how many typical fluctuations above or below average the cell sits. The field $\nu$ is the input of every count in the rest of the tutorial.


In [ ]:
cw.plot_points_to_field(n=128, growth=1.0)

Let's apply this to a larger dataset: the same construction at the full size, or the density cube of a simulation when its file is present.

Then we apply a threshold level $\nu_0$ and keep every cell with $\nu \ge \nu_0$. This mask allows us to understand what structures form the web.

---

**Quick Docs**

- `cw.load_dataset()`: `(nu, lam, label)`; `lam` is the parameter $\lambda$ of the Gaussian formula in §6.


In [ ]:
nu, lam, label = cw.load_dataset()
print(f'nu: shape {nu.shape}, dtype {nu.dtype}, mean {nu.mean():.2f}, scatter {nu.std():.2f}')
cw.plot_slices(nu, thresholds=[2, 0, -1])

What do we observe?

- $\nu \ge 2$: only the peaks survive (knots).
- $\nu \ge 0$: roughly half the cells survive as a connected network of sheets and filaments.
- $\nu \ge -1$: nearly everything survives, leaving isolated holes (voids).

The threshold changes how the set is connected. This tutorial measures that connectivity as a function of the level.


### 3. Why this problem

Counting how a set is connected sounds like a global problem: find every piece, tunnel and cavity. However, the quantity we compute, the *Euler characteristic*, decomposes it into a local one: a sum over grid cells that each look only at their immediate neighbours. How?

- Apply a 3D stencil, followed by a histogram (both common patterns in HPC):
    - The stencil reads each cell's neighbours.
    - The histogram adds one count per cell to a table.
- The result is an integer, which can be checked for exactness without tolerances.
- One pass over the field gives the whole curve.
- For a Gaussian random field the curve is known in closed form (§6): an independent check that our code is correct before we benchmark it.


### 4. Algorithm

#### Euler's formula on a pixel grid

Pick a level on a single plane and keep the pixels at or above it. On that mask:

- Call every pixel at or above the level a *vertex*.
- Where two vertices sit side by side or one above the other, join them with an *edge*.
- Where four vertices form a 2×2 block, fill the square between them with a *face*.

$$\chi = V - E + F$$

This number $\chi$ is the Euler characteristic of the mask: the number of connected pieces minus the number of holes in them. Vertices, edges and faces are the *cells* of the mask.

Let's count three small masks, then a larger one whose cells the animation draws while the counters tick.


In [ ]:
def euler_2d(mask):
    """(V, E, F, V - E + F) of a pixel mask: pixels, side-by-side pairs, 2x2 blocks."""
    v = mask.sum()
    e = (mask[1:, :] & mask[:-1, :]).sum() + (mask[:, 1:] & mask[:, :-1]).sum()
    f = (mask[1:, 1:] & mask[:-1, 1:] & mask[1:, :-1] & mask[:-1, :-1]).sum()
    return v, e, f, v - e + f

masks = {
    'one pixel': np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=bool),
    '2x2 block': np.array([[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]], dtype=bool),
    'ring':      np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=bool),
}
print(f"{'mask':<10} {'V':>3} {'E':>3} {'F':>3} {'V-E+F':>6}")
for name, m in masks.items():
    v, e, f, chi = euler_2d(m)
    print(f"{name:<10} {v:>3} {e:>3} {f:>3} {chi:>6}")

In [ ]:
mask = np.array([
    [0, 1, 1, 1, 0, 0],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
], dtype=bool)
print('6x6 mask: V-E+F =', euler_2d(mask)[3])
cw.animate_euler_buildup(mask)

- The 6×6 mask: the ring counts 0, the bar and the block 1 each, so three pieces with one hole give $\chi = 2$.

#### The curve on one plane

Now let's lower the level across the whole plane and compute $\chi$ at every step. This is the curve the tutorial measures, on one plane first.


In [ ]:
slice64 = nu[:64, :64, nu.shape[2] // 2]
cw.animate_flood(slice64, start=2.5, end=-2.5, interval=0.125)

- At the top level the mask is a few blobs and $\chi$ counts them.
- As the level falls the pieces merge and holes open inside them: $\chi$ rises, then falls back.

#### One more dimension

In 3D a grid cell is a *voxel*, and the mask has one more kind of cell to count. Where eight vertices form a 2×2×2 block, we fill the *cube* between them, and it enters the sum with a minus sign:

$$\chi = V - E + F - C.$$

- For a solid shape the sum counts pieces minus tunnels plus enclosed cavities: a ball gives 1, a ring 0, a hollow shell 2.
- Our box is periodic: a voxel on one side has neighbours on the opposite side, so the full box has no boundary and $\chi = 0$ once every voxel is in.
- Every voxel *owns* eight cells: its vertex, the edges to its +x, +y and +z neighbours, the three faces spanned by pairs of those edges, and the cube spanned by all three. Each cell extends in the positive directions from exactly one voxel, so every cell has one owner.
- A cell is present once all of its corners are at or above the level, so it first appears when the falling level reaches the lowest of its corners: that value is its *birth*.


In [ ]:
cw.isosurfaces(nu, thresholds=[2, 0, -1], sub=48)

- The three pictures of §2 in 3D: isolated blobs at the high level, one connected network in the middle, a solid with isolated holes at the low level.

#### Births

As the level falls, a cell enters the set at its birth and never leaves, so it counts at every level at or below its birth. Count each cell once:

- Let every cell cast one *token* at its birth: $+1$ for a vertex or a face, $-1$ for an edge or a cube.
- Sort the tokens into $K$ bins by birth level.
- $\chi$ at a level is the sum of the tokens in that bin and in every bin above it:

$$\chi(\nu) = \sum_{\text{cells}} (-1)^{\dim}\,[\,\text{birth} \ge \nu\,],$$

where the bracket is 1 when its condition holds and 0 otherwise. One pass over the field fills the bins, and one cumulative sum from the top turns the bins into the whole curve. The full box has $\chi = 0$, so all the tokens sum to zero: our first consistency check.

A hand-made 3×3 patch with values 0 to 3, one bin per value, makes the tokens concrete. Each level is checked against a direct count of $V - E + F$.

---

**Quick Docs**

- `cw.plot_birth_example(patch, k)`: every cell labelled with its birth bin, the tokens per bin, $\chi$ at every level; returns `(tokens per bin, chi)`.


In [ ]:
patch = np.array([[3, 1, 3],
                  [1, 0, 1],
                  [3, 1, 3]])
tokens, chi_patch = cw.plot_birth_example(patch, k=4)
for level in range(3, -1, -1):
    v, e, f, chi_direct = euler_2d(patch >= level)
    print(f'level {level}: chi from tokens {chi_patch[level]:>2}   direct V - E + F = {v} - {e} + {f} = {chi_direct}')

- Level 1: the four midpoints join, each with two edges to the corners next to it. Four $+1$ and eight $-1$ tokens, a net $-4$: $\chi = 0$, a ring around the centre.
- Level 0: the centre joins with its four edges and the four faces, a net $+1$: one solid piece, $\chi = 1$.

The same on a small cube, level by level: the set on the left, the tokens born in the current bin in the middle, the curve so far on the right.


In [ ]:
cw.animate_birthdays(nu[:8, :8, :8], k=16)

- The top bin holds a larger share of the vertices than of the cubes: a cell is born at the lowest of its corners, so it is born at or below every one of its vertices.
- Edges arrive once neighbouring voxels are both in. Faces and cubes come last.

### 5. Reference

`cw.ecc_hist_cupy(q, k)` is the reference, a dozen lines of CuPy: NumPy expressions over device arrays.

- Input: the field binned into integers. `cw.quantise` maps $\nu$ to a bin index $q$ in $[0, K)$ once, on the device, and every tool reads $q$.
- Output: the token histogram, one integer per bin.

```py
def ecc_cells_cupy(q):
    """Births of the eight cells each voxel owns, on the device: (vertex, (ex, ey, ez), (fxy, fxz, fyz), cube)."""
    ex = cp.minimum(q, cp.roll(q, -1, 0))
    ey = cp.minimum(q, cp.roll(q, -1, 1))
    ez = cp.minimum(q, cp.roll(q, -1, 2))
    fxy = cp.minimum(ex, cp.roll(ex, -1, 1))
    fxz = cp.minimum(ex, cp.roll(ex, -1, 2))
    fyz = cp.minimum(ey, cp.roll(ey, -1, 2))
    c = cp.minimum(fxy, cp.roll(fxy, -1, 2))
    return q, (ex, ey, ez), (fxy, fxz, fyz), c

def ecc_hist_cupy(q, k=K_BINS):
    """The reference: signed histogram of cell births (int64, length k) on the device, +1 per vertex and face, -1 per edge and cube."""
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_cupy(q)
    bc = lambda a: cp.bincount(a.ravel(), minlength=k)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)
```

The body is three blocks:

- **Neighbours.** `cp.roll` by $-1$ along an axis puts each voxel's $+1$ neighbour at the voxel's own position, wrapping around the periodic box.
- **Births.** `cp.minimum` takes the element-wise minimum of two arrays. An edge is born at the lower of its two ends. A face is born at the lower of an edge and that edge shifted one step along the face's other axis. The cube is born at the lower of a face and the face one step above it.
- **Histogram.** `cp.bincount` counts how many births of one cell array fall into each of the $K$ bins. The signed sum of the eight counts is the token histogram.

`cw.ecc_from_hist` turns the histogram into $\chi$ at every bin edge with one cumulative sum from the top. Binning preserves order, so the births computed from $q$ are the binned births of $\nu$: the histogram is exact, and every tool has to reproduce it bin for bin.

Let's run the reference on the cube of §2.

---

**Quick Docs**

- `cw.quantise(nu)`: the bin index of every voxel, int32, on the device that holds `nu`.
- `cw.bin_edges()`: the lower edge of every bin.
- `cw.ecc_hist_cupy(q)`: the token histogram, the reference every tool is checked against.
- `cw.ecc_from_hist(hist)`: $\chi$ at every bin edge.


In [ ]:
K = cw.K_BINS
q = cw.quantise(cp.asarray(nu))
edges = cw.bin_edges()
hist = cp.asnumpy(cw.ecc_hist_cupy(q))
chi = cw.ecc_from_hist(hist)
assert hist.sum() == 0, hist.sum()
print(f'N = {nu.shape[0]}^3, K = {K}: the tokens sum to {hist.sum()}, max |chi| = {np.abs(chi).max()}')
cw.plot_hist_curve(edges, hist, chi)

- The histogram is positive at high levels, where mostly vertices are born, and negative in the middle, where edges and cubes dominate.

### 6. Validation

The later notebooks time the counting on a *test field*, a Gaussian random field generated at any size. For a Gaussian random field the expected curve $\chi(\nu)$ is known in closed form ([Hamilton et al. 1986](https://doi.org/10.1086/164571)), so the measured curve has to follow it. Let's generate the test field at the size of our dataset and overlay the formula.

---

**Quick Docs**

- `cw.make_binned_field(n)`: `(q, lam)`, the binned test field (int32, on the device) and the parameter $\lambda$ the formula takes, computed from the field; cached under `data/`.
- `cw.chi_gaussian(edges, lam, volume)`: the expected $\chi$ at every bin edge.
- `cw.plot_overlay(edges, chi, pred)`: measured against expected with the residual underneath; returns `(relative residual, zero crossings)`.


In [ ]:
N = nu.shape[0]
q, lam = cw.make_binned_field(N)
chi = cw.ecc_from_hist(cp.asnumpy(cw.ecc_hist_cupy(q)))
pred = cw.chi_gaussian(edges, lam, q.size)
rel, zeros = cw.plot_overlay(edges, chi, pred, title=f'the test field at {N}^3 against the Gaussian expectation')
print(f'relative residual {rel:.3f} (tolerance {cw.ANALYTIC_TOL}), zero crossings at nu = {np.round(zeros, 2).tolist()}')
assert rel < cw.ANALYTIC_TOL, rel

- Both curves cross zero near $\nu = \pm 1$, where the formula puts the crossings for every Gaussian field.


### 7. Sizing

Let's fix the problem sizes from this machine's memory and measure the device's copy bandwidth. The bandwidth is the roof every later figure is set against: no kernel can move its bytes faster than a device-to-device copy moves them. Both go into `machine.json`, so every notebook sweeps the same sizes. A tool whose temporaries do not fit at a size skips it and says so.

---

**Quick Docs**

- `cw.save_sizing()`: measures the device copy bandwidth, a one GiB device-to-device copy timed with CUDA events, then fixes the canonical size (the size every notebook times), the sweep sizes and the device's facts in `machine.json`.


In [ ]:
cw.save_sizing()

### 8. Baseline

Let's time the reference at the canonical size and save the result as `timings/cupy.json`. Each later notebook writes its own file to the same directory. The timed region is the histogram only: the field and its binning are already on the device, and the cumulative sum stays on the host.

---

**Quick Docs**

- `cw.field(n)`: the binned field at size n on the device, built once per size, so no timed call pays for it.
- `cw.measure(stage, run, n, prepare, record)`: `run(n)` timed cold (the first call), then warm (the median of repeated calls); the gate, an exact integer comparison against the reference curve; one record line printed and, with `record`, written to the stage file `timings/<stage>.json`. `prepare(n)` runs before the timer.
- `cw.save_sweep(stage, fn)`: rates at every size the stage affords, each checked against the reference, attached to the stage file.


In [ ]:
N = cw.canonical_n()


def run_cupy(n):
    hist = cw.ecc_hist_cupy(cw.field(n), K)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist)


cw.measure('cupy', run_cupy, N)
cw.save_sweep('cupy', run_cupy)

Takeaways:

- One pass over the field fills the token histogram. One cumulative sum turns it into the whole curve.
- Two checks need no timing: the tokens sum to zero, and the test field's curve follows the Gaussian formula.

### 9. Ladder

Each later notebook implements `ecc_hist_<tool>` in its tool, on the field already on the device, and passes the gate: the same integers at every bin edge, and a histogram that sums to zero.

$$\max_b |\chi_{\text{tool}}[b] - \chi_{\text{cupy}}[b]| = 0 \quad\text{and}\quad \sum_b \text{hist}[b] = 0.$$

| NB | Tool | Programming-model hook |
|---|---|---|
| 91 | **CuPy** | Array model: every line launches a kernel and allocates a temporary |
| 92 | **JAX** | Traced graph: XLA fuses the chain |
| 93 | **cuda.compute** | Algorithm over iterators: the stencil evaluated inside the library's kernel |
| 94 | **CppJIT** | C++ in the session: CUB's histogram, then one CUDA kernel |

`95__cosmic_web__synthesis.ipynb` ends the block: the curves, the timelines and the bytes each model moved, built from `timings/`.

### 10. Conventions

Each exercise (91-94) comes as `NN__topic__subtopic.ipynb`, the notebook we run live, with `# TODO:` cells whose body is `...`, and `solutions/NN__topic__subtopic__SOLUTION.ipynb`, the complete notebook. A **Quick Docs** block precedes the code that uses new APIs. `# EXTRA CREDIT:` cells are optional.

Each notebook runs in its own Jupyter kernel and holds its arrays until that kernel is shut down. Shut down the previous notebook's kernel before you run the next one. Otherwise its first cell stops and names the memory it lacks.

Next: `91__cosmic_web__cupy__array_model.ipynb` traces one call of the reference and counts what each line costs.
